# Generación de Código con Gemini

En este cuaderno usaremos el modelo **Gemini 3.5 Flash** de Google para generar
código Python a partir de instrucciones en lenguaje natural.

### ¿Qué aprenderemos?
- Cómo usar Gemini para generar código Python automáticamente.
- Cómo configurar parámetros como `temperature` y `max_output_tokens`.
- Cómo crear una función reutilizable para generación de código.

### Requisitos previos
- Tener configurado el secreto `GOOGLE_API_KEY` en Colab.


## Paso 1: Configurar la API Key

In [3]:
from google.colab import userdata
import os

# Obtener la API key desde los secretos de Colab
api_key = userdata.get('GOOGLE_API_KEY')
os.environ['GOOGLE_API_KEY'] = api_key

print("API Key cargada:", "Sí" if api_key else "No")

API Key cargada: Sí


## Paso 2: Instalar e importar librerías

In [4]:
# Instalar el SDK oficial de Google GenAI (el SDK antiguo google-generativeai ya está descontinuado)
!pip install -q -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.4.0 which is incompatible.


In [5]:
from google import genai
from google.genai import types

# Crear el cliente con la API Key
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

print("Cliente de Gemini creado exitosamente!")

Cliente de Gemini creado exitosamente!


## Paso 3: Definir la función para generar código

Creamos una función que toma un prompt en lenguaje natural y usa Gemini
para generar código Python.

### Parámetros de configuración:
- `temperature`: controla la creatividad (baja = más predecible, alta = más variada). Para los modelos Gemini 3, Google recomienda dejar el valor por defecto (`1.0`), porque su razonamiento está optimizado para ese valor; puedes experimentar con otros, pero el resultado puede variar.
- `max_output_tokens`: longitud máxima de la respuesta. Gemini 3 "piensa" antes de responder y esos tokens de razonamiento cuentan dentro del límite, por eso usamos un valor amplio.
- `system_instruction`: instrucción del sistema que define el comportamiento del modelo.

In [8]:
def generar_codigo_gemini(prompt, temperatura=1.0):
    """
    Genera código Python usando el modelo Gemini 3.5 Flash.

    Args:
        prompt (str): Descripción en lenguaje natural del código a generar.
        temperatura (float): Creatividad del modelo (0.0 a 2.0). Por defecto 1.0, el valor recomendado para Gemini 3.

    Returns:
        str: Código Python generado.
    """
    try:
        respuesta = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction="Eres un programador experto en Python. Genera código limpio, bien comentado y funcional.",
                temperature=temperatura,
                max_output_tokens=8192,
            )
        )
        return (respuesta.text or "").strip()

    except Exception as e:
        return f"Error en la generación: {str(e)}"

## Paso 4: Ejecutar la función con prompts de ejemplo

Probamos la generación de código con dos problemas clásicos de programación.

In [9]:
# Prompt 1: Fibonacci con programación dinámica
prompt_fibonacci = "Crear una función en Python para calcular los números fibonacci usando programación dinámica."

codigo_fibonacci = generar_codigo_gemini(prompt_fibonacci)
print("Código para Fibonacci usando programación dinámica:\n")
print(codigo_fibonacci)
print("\n" + "_" * 60 + "\n")

# Prompt 2: Árbol de Búsqueda Binaria
prompt_bst = "Implementar en Python una clase de Árbol de Búsqueda Binaria con métodos insert y search."

codigo_bst = generar_codigo_gemini(prompt_bst)
print("Código para clase Binary Search Tree:\n")
print(codigo_bst)

Código para Fibonacci usando programación dinámica:

Aquí tienes una solución completa en Python para calcular el $n$-ésimo número de Fibonacci utilizando **Programación Dinámica**. 

Te presento dos enfoques comunes: 
1. **Bottom-Up (Tabulación / Optimizado en espacio)**: El más eficiente en memoria y tiempo.
2. **Top-Down (Memorización)**: Basado en recursividad guardando resultados previos.

---

### Opción 1: Enfoque Bottom-Up (Recomendado - Optimizado en Espacio)

Este enfoque construye la solución desde el caso base ($0$ y $1$) hasta $n$. Como solo necesitamos los dos números anteriores para calcular el siguiente, optimizamos el espacio a $O(1)$.

```python
def fibonacci_dp(n: int) -> int:
    """Calcula el n-ésimo número de Fibonacci usando Programación Dinámica (Bottom-Up).

    Complejidad Temporal: O(n)
    Complejidad Espacial: O(1)

    :param n: Posición del número de Fibonacci deseado (n >= 0).
    :return: El n-ésimo número de Fibonacci.
    """
    # Validación de entra